In [5]:
!pip install mlxtend
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules




In [14]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

df = pd.read_excel('/content/preprocessed_depression.csv.xlsx')
df_cleaned = df.drop(columns=[col for col in df.columns if 'Unnamed' in col])
df_cleaned = df_cleaned.astype(str)

#Convert to transactions
transactions = df_cleaned.apply(lambda row: [f"{col}={val}" for col, val in row.items()], axis=1).tolist()

#Encode
te = TransactionEncoder()
te_ary = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

#Apriori for getting frequent itemsets
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

#Generate all association rules
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

#filter rules with ≥2 items in antecedent
rules['antecedent_len'] = rules['antecedents'].apply(lambda x: len(x))
rules = rules[rules['antecedent_len'] >= 2]

#Filter rules where Depression=1 or Depression=0 is the CONSEQUENT
rules_depression_as_conseq = rules[rules['consequents'].astype(str).str.contains('Depression=1|Depression=0')].copy()

#Clean up the display
rules_depression_as_conseq['antecedents'] = rules_depression_as_conseq['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
rules_depression_as_conseq['consequents'] = rules_depression_as_conseq['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

rules_conf_table = rules_depression_as_conseq[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules_conf_table = rules_conf_table.sort_values(by=['confidence', 'lift'], ascending=False)

#Display top patterns
rules_conf_table.head(10)

,antecedents,consequents,support,confidence,lift
14308,"Academic Pressure=High, Dietary Habits=Unhealt...",Depression=1,0.010290,0.996528,1.701883
11764,"Academic Pressure=High, Dietary Habits=Unhealt...",Depression=1,0.014843,0.995192,1.699602
12464,"Academic Pressure=Low , Dietary Habits=Healthy...",Depression=0,0.010792,0.993399,2.396877
14314,"Academic Pressure=High, Family History=No, Fin...",Depression=1,0.010720,0.993355,1.696465
14264,"Academic Pressure=High, Dietary Habits=Unhealt...",Depression=1,0.010361,0.993127,1.696075
12188,"Academic Pressure=High, Financial Stress=High,...",Depression=1,0.022408,0.992063,1.694259
11774,"Academic Pressure=High, Dietary Habits=Unhealt...",Depression=1,0.022085,0.991948,1.694062
12458,"Academic Pressure=Low , Dietary Habits=Healthy...",Depression=0,0.012799,0.991667,2.392696
11761,"Academic Pressure=High, Dietary Habits=Unhealt...",Depression=1,0.012046,0.991150,1.692699
11531,"Academic Pressure=High, Dietary Habits=Moderat...",Depression=1,0.015739,0.990971,1.692392
